#  **02. Data Preprocessing & Feature Engineering**

## Tổng quan (Overview)
Notebook này tập trung vào việc chuẩn bị dữ liệu (Data Preparation) nối tiếp sau bước Khám phá dữ liệu (EDA). Mục đích nhằm biến đổi dữ liệu về trạng thái tối ưu nhất dành cho các mô hình Machine Learning.

- **Đầu vào (Input):** File dữ liệu thô hoặc đã qua làm sạch cơ bản.

- **Đầu ra (Output):** Bộ dữ liệu hoàn chuẩn bị cho mô hình học máy.

##  Bố cục Pipeline (Table of Contents)
* [0. Cài đặt Môi trường & Load dữ liệu](#0)
* [1. Tiền xử lý dữ liệu - Giai đoạn 1 (Làm sạch cơ sở & Hợp nhất)](#1)
    * [1.1 Bước 1: Quản trị không gian đặc trưng & biến định danh (Drop IDs, Name, Tách Target)](#1-1)
    * [1.2 Bước 2: Chuẩn hóa văn bản nhiễu trước khi Encode (Normalize Text)](#1-2)
    * [1.3 Bước 3: Hợp nhất đặc trưng MNAR](#1-3)
* [2. Chuẩn hóa đặc trưng (Feature Engineering)](#2) 
    * [2.1 FE-1: Tổng hợp ma trận áp lực đa chiều](#2-1)
    * [2.2 FE-2: Hệ số hao mòn sinh học & thâm hụt thời gian](#2-2)
    * [2.3 FE-3: Phân rã và xác định cờ nhóm tuổi nguy cơ cao](#2-3)
    * [2.4 FE-4: Cấu trúc điểm rủi ro lâm sàng phức hợp](#2-4)
    * [2.5 FE-5: Chỉ số hài lòng tổng hợp](#2-5)
    * [2.6 FE-6 (Tùy chọn): Khai thác NLP (TF-IDF & SVD) cho Profession & Degree](#2-6)
* [3. Tiền xử lý dữ liệu - Giai đoạn 2 (Xử lý Missing & Encoding)](#3)
    * [3.1 Bước 4: Xử lý giá trị thiếu (NaN) còn sót lại (Theo tư duy từng Model)](#3-1)
    * [3.2 Bước 5: Ordinal Encoding (Cho các biến có tính thứ tự)](#3-2)
    * [3.3 Bước 6: One-Hot Encoding (Cho các biến danh nghĩa <= 5 nhãn)](#3-3)
    * [3.4 Bước 7: Smoothed Target Encoding (Xử lý High-Cardinality với Cross-Val)](#3-4)
    * [3.5 Bước 8: Chuẩn hóa không gian độ lượng (Scaling độc lập cho Neural Networks)](#3-5)
* [4. Tối ưu Đánh giá và Mất cân bằng lớp (Validation & Class Imbalance)](#4)
    * [4.1 Bước 9: Phân rã Cross-Validation (chuẩn bị hạ tầng Stacking)](#4-1)
    * [4.2 Bước 10: Trọng số nội tại (Scale Pos Weight & Không gian Optuna)](#4-2)
* [5. Xuất và Lưu trữ dữ liệu chuẩn (Export to `.pkl` / `.csv`)](#5)


# 0. Cài đặt môi trường và load dữ liệu

In [8]:
pip install -r ../requirements.txt


  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached statsmodels-0.14.6-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached jupyter_server-2.17.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached jupyterlab_server-2.28.0-py3-none-any.whl.metadata (5.9 kB)
  Using cached jupyterlab-4.5.6-py3-none-any.whl.metadata (16 kB)
  Using cached notebook_shim-0.2.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached argon2_cffi-25.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached jupyter_events-0.12.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached jupyter_server_terminals-0.5.4-py3-none-any.whl.metadata (5.9 kB)
  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached pywinpty-3.0.3-cp313-cp313-win_amd64.whl.metadata (5.9 kB)
  Using cached send2trash-2.1.0-py3-none-any.whl.metadata (4.1 kB)
  Using cached terminado-0.18.1-py3-none-any.whl.metadata (5.8 kB)
  Using cach

In [7]:
import os 
import warnings
from pathlib import Path 
import numpy as np 
import pandas as pd 

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)   

#================ THIẾT LẬP ĐƯỜNG DẪN =======================
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Thư mục Root của Project: {PROJECT_ROOT}")


# ================= CẤU HÌNH CÁC THAM SỐ ====================
class Config:
    # --- Trạng thái Seed hệ thống (Đảm bảo Reproducibility) ---
    RANDOM_STATE = 42
    
    # --- Cấu hình Machine Learning ---
    N_FOLDS = 5                 # [Bước 9] Phân rã K-Fold Cross Validation
    TREE_MISSING_VAL = -1       # [Bước 4] Giá trị missing cho XGBoost/LightGBM
    NN_IMPUTE_NEIGHBORS = 5     # [Bước 4] k của KNNImputer cho mạng Neural
    
    # --- Cấu hình Tên cột cốt lõi ---
    TARGET_COL = "Depression"
    DROP_COLS = ["id", "Name"]  # [Bước 1] Các cột rác gây bùng nổ chiều
    
    # --- Cấu hình file Nguồn và Đích ---
    TRAIN_INPUT = RAW_DATA_DIR / "train.csv"
    TEST_INPUT = RAW_DATA_DIR / "test.csv"
    
    # Các file Output cuối cùng (Cho Bước 11)
    TRAIN_TREE_OUT = PROCESSED_DATA_DIR / "train_tree_ready.pkl"
    TEST_TREE_OUT = PROCESSED_DATA_DIR / "test_tree_ready.pkl"
    
    TRAIN_NN_OUT = PROCESSED_DATA_DIR / "train_nn_ready.pkl"
    TEST_NN_OUT = PROCESSED_DATA_DIR / "test_nn_ready.pkl"
config = Config()

# ============================= LOAD DATASET =====================
try:
    train_df = pd.read_csv(config.TRAIN_INPUT)
    test_df = pd.read_csv(config.TEST_INPUT)
    
    print("-" * 50)
    print(f"Tải dữ liệu thành công!")
    print(f"Tập Train (Row, Col): {train_df.shape}")
    print(f"Tập Test  (Row, Col): {test_df.shape}")
    print("-" * 50)
except FileNotFoundError as e:
    print(f"LỖI KHÔNG TÌM THẤY DỮ LIỆU. Vui lòng kiểm tra lại thư mục /data/raw! Chi tiết lỗi: {e}")



Thư mục Root của Project: d:\2025-2026 HKII\Data Analystic\Project\Exploring-Mental-Health-Data
--------------------------------------------------
Tải dữ liệu thành công!
Tập Train (Row, Col): (140700, 20)
Tập Test  (Row, Col): (93800, 19)
--------------------------------------------------


# 1. Tiền xử lý dữ liệu - Giai đoạn 1 (Làm sạch cơ sở & Hợp nhất)

### 1.1 Bước 1: Quản trị không gian đặc trưng & biến định danh (Drop IDs, Name, Tách Target)
Ở bước đầu tiên, chúng ta sẽ làm sạch cấu trúc khung DataFrame gốc nhằm chuẩn bị cho việc encode và tránh các rủi ro chết người trong Machine Learning:
1. **Tách mục tiêu (Target) `Depression`**: Bắt buộc phải tách ngay `y` ra khỏi `X_train`. Nếu giữ và xử lý chung, chúng ta rất dễ mắc phải lỗi nghiêm trọng là rò rỉ dữ liệu (Data Leakage) khi tính toán các Imputer hoặc Target Encoding.

2. **Loại bỏ cột `id`**: Tương tự như Index, `id` chỉ là ID hệ thống, không có phương sai dự báo thực tế về tình trạng trầm cảm. Tuy nhiên, ta cần **giữ lại ID của tập Test** để xuất báo cáo/sumit mô hình cuối cùng.

3. **Loại bỏ cột `Name`**: Biến này có độ phân mảnh (cardinality) rất khổng lồ. Tên con người không mang tín hiệu dự báo cho sức khỏe tâm thần. Giữ lại sẽ dẫn tới số chiều tăng lên đáng kể (Curse of Dimensionality) và Overfitting thảm hoạ nếu bị One-Hot Encode.

In [10]:
def initial_data_cleanup(train, test, config): 
    '''
        Tách nhãn Target từ tập Train, lưu trữ test_id và loại bỏ các cột định danh
        Hàm đảm bảo sự đồng bộ số lượng cột và kích thước giữa 2 tập Train/Test
    '''
    y_train = train[config.TARGET_COL].copy()
    X_train = train.drop(columns=[config.TARGET_COL])  # xóa y 
    X_test = test.copy()

    test_identify = X_test['id'].copy() if 'id' in X_test.columns else None # copy id của tập test 

    X_train = X_train.drop(columns=config.DROP_COLS, errors="ignore")
    X_test = X_test.drop(columns=config.DROP_COLS, errors="ignore")
    print(f"Shape X_train: {X_train.shape} | Shape y_train {y_train.shape}")
    print(f"Shape X_test {X_test.shape} | Lưu test_id: {len(test_identify)} rows")
    return X_train, y_train, X_test, test_identify 

# ================= CHẠY HÀM =====================
X_train, y_train, X_test, test_identify = initial_data_cleanup(train_df, test_df, config)
assert X_train.shape[1] == X_test.shape[1], "Kích thước cột Train và Test đang lệch nhau!"
display(X_train.head(3))



Shape X_train: (140700, 17) | Shape y_train (140700,)
Shape X_test (93800, 17) | Lưu test_id: 93800 rows


,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness
0,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No
1,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No
2,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No


### 1.2 Bước 2: Chuẩn hóa văn bản nhiễu trước khi Encode (Normalize Text)
Trong các bộ dữ liệu do người dùng tự điền (Survey/Form), các cột thuộc tính phân loại (categorical) thường xuyên mắc lỗi "dirty text" như: viết hoa/nhỏ lộn xộn, dư khoảng trắng, hoặc có vô vàn biến thể đồng nghĩa (Ví dụ: `less healthy`, `less than healthy`, `moderate` thực chất cùng chỉ mức độ "trung bình").
**Lý do tại sao phải xóa nhiễu TRƯỚC bước Cấu trúc và Encode:**
- Nếu không gom cụm các biến thể đồng nghĩa, các bộ Encode (Ordinal/One-Hot) về sau sẽ xem tất cả chúng là các đối tượng riêng biệt → Làm phình to số chiều và làm nhiễu độ chính xác học của Cây quyết định (Tree splits).

- **Mục tiêu bước này:** Rút gọn toàn bộ cột `Dietary Habits` từ một mớ biến thể hỗn loạn tụm lại đúng 3 nhãn sạch sẽ, chặt chẽ: `"unhealthy"`, `"moderate"`, và `"healthy"`.

In [13]:
def normalize_dietary_text(val): 
    '''
        Gom các biến thể từ nhãn nhiễu về nhãn chuẩn
    '''
    if pd.isna(val): 
        return val # nếu là NaN thì xử lý = Imputer 
    v = str(val).lower().strip()
    # nhóm ko lành mạnh 
    if v in ["no healthy", "unhealthy"]: 
        return "unhealthy"
    
    # nhóm tbinh 
    if v in ["less_healthy", "less than healthy", "moderate"]: 
        return "moderate"
    
    # nhóm lành mạnh 
    if "healthy" in v: 
        return "healthy"
    
    return np.nan # vì có những từ người dùng điền ko hợp lệ  

def apply_text_normalization(X_train, X_test):
    '''
        Trình bao bọc (Wrapper) áp dungh hàm apply lên toàn bộ dữ liệu
    '''
    X_tr = X_train.copy()
    X_te = X_test.copy()

    col = 'Dietary Habits'
    if col in X_tr.columns: 
        X_tr[col] = X_tr[col].apply(normalize_dietary_text)
        X_te[col] = X_te[col].apply(normalize_dietary_text)
    print(f"Đã chuẩn hóa xong. Kiểm tra trên tập Train: ")
    print(X_tr[col].value_counts(dropna=False).to_frame()) 

    return X_tr, X_te 
# ============== CHẠY HÀM =================
X_train, X_test = apply_text_normalization(X_train, X_test)



Đã chuẩn hóa xong. Kiểm tra trên tập Train: 
                count
Dietary Habits       
moderate        49706
unhealthy       46228
healthy         44744
NaN                22


### 1.3 Bước 3: Hợp nhất đặc trưng MNAR (Thay thế Median Imputation)

**Hiểm họa của Imputation mù quáng:**
Trong tập dữ liệu, các cột như `Academic Pressure` (Áp lực học tập) hay `Work Pressure` (Áp lực công việc) có tỷ lệ Null khổng lồ lên tới 20% - 80%. Nguyên nhân không phải do dữ liệu bị mất, mà là do cơ chế **độc quyền lẫn nhau (Mutually Exclusive)**: Học sinh thì không phải đi làm, và người đi làm thì không có áp lực ở trường.
Nếu sử dụng `Median_Imputer` điền trung vị  vào đây, ta vô tình bắt một "Người đi làm" gánh thêm con điểm 3.5 áp lực học tập của "Sinh viên". Thuật toán sẽ học ra những Pattern sai lệch thực tế!
Mục tiêu của hàm này là **Trộn chéo (Consolidate)** các cột rời rạc đó lại thành 2 biến vĩ mô phản ánh tổng quát trục Tâm lý học: 
1. `Overall_Pressure`: Áp lực tổng quan.
2. `Overall_Satisfaction`: Chỉ số hài lòng tổng quan.

In [15]:
# =====================================================================
# BẢNG KIỂM CHỨNG TỶ LỆ KHUYẾT THIẾU (MISSING REPORT)
# =====================================================================
def get_missing_report(df, title="BÁO CÁO TỶ LỆ RỖNG (NaN)"):
    null_counts = df.isnull().sum()
    null_percentages = (null_counts / len(df)) * 100
    missing_df = pd.DataFrame({
        'Count (Số lượng NaN)': null_counts,
        'Percentage (%)': null_percentages
    })
    missing_df = missing_df[missing_df['Count (Số lượng NaN)'] > 0]
    missing_df = missing_df.sort_values(by='Percentage (%)', ascending=False)
    
    print("-" * 50)
    print(title)
    print("-" * 50)
    
    if missing_df.empty:
        print("Dataset không có ô rỗng.")
    else:
        display(missing_df.style.format({'Percentage (%)': "{:.2f}%"}))
        

get_missing_report(train_df, title="KHẢO SÁT TỶ LỆ DỮ LIỆU RỖNG TRÊN TẬP TRAIN GỐC")


--------------------------------------------------
KHẢO SÁT TỶ LỆ DỮ LIỆU RỖNG TRÊN TẬP TRAIN GỐC
--------------------------------------------------


,Count (Số lượng NaN),Percentage (%)
Academic Pressure,112803,80.17%
Study Satisfaction,112803,80.17%
CGPA,112802,80.17%
Profession,36630,26.03%
Work Pressure,27918,19.84%
Job Satisfaction,27910,19.84%
Dietary Habits,4,0.00%
Financial Stress,4,0.00%
Degree,2,0.00%


In [21]:
def consolidate_mnar_features(df): 
    '''
         Kéo dữ liệu từ các cột áp lực và hài lòng rời rạc, gộp thành 2 cột tâm lý thống nhất dựa trên cờ hành vi 'Working Professional' hoặc 'Student' 
    '''
    df = df.copy()
    df['Overall_Pressure'] = np.nan 
    df['Overall_Satisfaction'] = np.nan 

    # sinh viên 
    mask_student = df['Working Professional or Student'] == 'Student'
    df.loc[mask_student, 'Overall_Pressure'] = df.loc[mask_student, 'Academic Pressure']
    df.loc[mask_student, 'Overall_Satisfaction'] = df.loc[mask_student, 'Study Satisfaction']

    # người đi làm 
    mask_worker = df['Working Professional or Student'] == 'Working Professional'
    df.loc[mask_worker, 'Overall_Pressure'] = df.loc[mask_worker, 'Work Pressure']
    df.loc[mask_worker, 'Overall_Satisfaction'] = df.loc[mask_worker, 'Job Satisfaction']

    # sau gộp thì xóa các cột 
    cols_to_drop = [
        'Academic Pressure', 'Work Pressure', 
        'Study Satisfaction', 'Job Satisfaction', 'Profession', 'CGPA', 'Degree'
    ]
    df = df.drop(columns=cols_to_drop, errors='ignore')
    return df

def apply_consolidation(X_train, X_test): 
    X_train = consolidate_mnar_features(X_train)
    X_test = consolidate_mnar_features(X_test)
    print(f"Rút gọn thành công. Khung dữ liệu hiện tại còn: {X_train.shape[1]} cột.")
    print(f"Số Missing trên cột Overall_Pressure trên tập Train: ", X_train['Overall_Pressure'].isnull().sum())

    return X_train, X_test 


# =================== CHẠY HÀM ======================
X_train, X_test = apply_consolidation(X_train, X_test)
display(X_train[['Working Professional or Student', 'Overall_Pressure', 'Overall_Satisfaction']].head(5))



Rút gọn thành công. Khung dữ liệu hiện tại còn: 12 cột.
Số Missing trên cột Overall_Pressure trên tập Train:  29


,Working Professional or Student,Overall_Pressure,Overall_Satisfaction
0,Working Professional,5.0,2.0
1,Working Professional,4.0,3.0
2,Student,5.0,2.0
3,Working Professional,5.0,1.0
4,Working Professional,1.0,1.0


#### **Nhận xét:**

- Hiện tại Overall_Pressure giảm còn 29 ô missing. Lấp đầy ô trống mà không cần tới các thuật toán ML. Cột `Overall_Pressure` giờ là thanh đo cho **Căng thẳng trong học tập** lẫn **Căng thẳng trong công việc**

- Con số `29` này có thể là do các nguyên nhân như: 
    
    - Học sinh vô tình bỏ trống 
    - Quên chọn thẻ "Student/Working" 

- Mức độ Missing bây giờ khá bé, để xử lý việc này chúng ta sẽ cho 29 người này vào thuật toán KNNImputer.

# 2. Chuẩn  hóa đặc trưng (Feature Engineering)

### 2.1 FE-1: Tổng hợp ma trận áp lực đa chiều (Sức nặng môi trường)
Áp lực tâm lý ít khi hoạt động đơn lẻ độc lập. Thông thường, sự giao thoa giữa **Áp lực làm việc/học tập** và **Áp lực tài chính** mới tạo ra đòn chí mạng. Thay vì để các mô hình Tree-based (XGBoost/CatBoost) tự gồng mình dò dẫm xây các luồng nhát cắt chéo, ta chủ động cấu trúc hóa không gian của chúng:
1. **Total_Stress_Sum (Tổng tuyến tính):** Cân đo sức nặng tổng thể.
2. **Stress_Resonance_Index (Độ cộng hưởng phi tuyến tính):** Đo lường **"Áp lực"**. Một người nghèo nhưng công việc rất nhàn nhã (Điểm tài chính 5 x Điểm áp lực công việc 1 = 5) sẽ ít nguy cơ trầm cảm hơn người vừa hết tiền vừa bị sếp chửi ngập đầu (5 x 5 = đỉnh điểm 25). Phép nhân tạo ra vùng phân rẽ rõ ràng giúp cây quyết định phân loại cực bén.